# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sanaullah-Turab/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Lane 2 (Refresh / Content Opportunity Scoring) is a **ranking / scoring** problem at the top level: the final output is an ordered queue of pages, not a single label. But it's built on top of a **classification** sub-task, the way the starter pipeline does it: a model first predicts the probability that a page is declining (`is_declining_label`), then that probability gets blended with a transparent baseline score into `final_refresh_score`, which is what actually ranks the queue.

So the honest answer is: classification underneath, ranking/scoring on top. The metric that matters lives at the ranking level (precision@K), not at raw classification accuracy, because the review team never sees a bare 0/1 label, they see an ordered list with a fixed number of slots they can act on.


In [1]:
task_framing = {
    "top_level_task": "ranking / scoring",
    "sub_task": "classification (probability of decline)",
    "why_ranking_not_pure_classification": "reviewers act on an ordered queue with fixed capacity, not on isolated 0/1 labels",
}
task_framing


{'top_level_task': 'ranking / scoring',
 'sub_task': 'classification (probability of decline)',
 'why_ranking_not_pure_classification': 'reviewers act on an ordered queue with fixed capacity, not on isolated 0/1 labels'}

## 2. Target or proxy

The starter target is `is_declining_label = (trend_direction == "down")`. This is a **proxy**, not an observed future outcome: `trend_direction` is itself derived from `trend_pct`, computed from the current 90-day window. That means the label describes what already happened by the time you'd be predicting it, not what happens next. Because of that, `trend_direction` and `trend_pct` can never be used as features, only as the label, or the model would just learn to read its own answer back.

This proxy is fine for a first pass (it's exactly how the starter pipeline gets its first honest numbers), but it's a known weakness I'm carrying forward, not hiding. A stronger version of this target, once I move to the warehouse release, would be a genuine future-window label: prior 90 days of features predicting decline or recovery over the next 30 days, so the model predicts something that hasn't happened yet at prediction time.


In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

trend_counts = df["trend_direction"].value_counts()
declining_rate = (df["trend_direction"] == "down").mean()

print(trend_counts)
print(f"\nproxy label 'is_declining_label' rate: {declining_rate:.1%}")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

proxy label 'is_declining_label' rate: 54.2%


## 3. Success metric

**Precision@50**: of the top 50 pages the ranked queue puts forward, how many are actually declining-with-demand or otherwise flagged by the time a reviewer checks them. I'm picking 50 because that's roughly what the lane guide frames as a realistic weekly review capacity, and because the starter pipeline already reports this exact metric, so my number is directly comparable to the baseline (0.240) and the trained model (0.740) from `outputs/model_report.md`.

I'm not using raw accuracy or a single global ROC-AUC as the headline number, because a reviewer never looks at the whole dataset, only the top slice their capacity allows. A metric that scores the whole list equally would reward the model for getting page 10,000 right just as much as page 1, which isn't how this gets used. I'll keep recall on the declining set as a secondary check, so the top-K focus doesn't quietly starve real problem pages further down the queue.


In [3]:
base_rate = declining_rate
k = 50

print(f"base rate of the proxy positive class: {base_rate:.1%}")
print(f"if precision@{k} just matched the base rate, that would mean no real lift over guessing")
print(f"the starter baseline rule scores 0.240 at precision@{k} -- below even a naive guess at the base rate")
print(f"the starter random forest scores 0.740 at precision@{k} -- clearly above it")


base rate of the proxy positive class: 54.2%
if precision@50 just matched the base rate, that would mean no real lift over guessing
the starter baseline rule scores 0.240 at precision@50 -- below even a naive guess at the base rate
the starter random forest scores 0.740 at precision@50 -- clearly above it


## 4. The unit of analysis, as a real dataframe

One row = one page (`content_id`), scoped to one client (`client_id`). Below is the lane's working slice: the columns a refresh-scoring model would actually use, pulled from the starter CSV.


In [4]:
lane_columns = [
    "content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
    "sessions_90d", "avg_position", "ctr", "engagement_rate", "scroll_rate",
    "word_count", "content_age_days", "days_since_last_update", "freshness_tier",
    "trend_direction", "trend_pct",
]

lane_slice = df[lane_columns]
print("rows:", len(lane_slice), "| one row = one page (content_id) for one client (client_id)")
lane_slice.head()


rows: 30000 | one row = one page (content_id) for one client (client_id)


,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,word_count,content_age_days,days_since_last_update,freshness_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,10.6,0.76,5.88,4.55,3221.0,187,20,0-30,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,20.3,0.05,0.00,10.00,2481.0,445,25,0-30,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,36.5,0.09,0.00,28.57,3515.0,141,20,0-30,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,6.2,0.49,1.28,3.45,NaN,463,22,0-30,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,44.0,0.13,0.00,24.29,2803.0,263,14,0-30,down,-34.7


## 5. Why ML beats a fixed rule here

The lane guide's baseline already IS a fixed rule (a weighted sum of five hand-picked flags), and it only gets precision@50 = 0.240. The gap to the trained model's 0.740 is the evidence: a single if-statement can't capture how these signals interact.

Concretely: five simple review flags (`stale_visible_page`, `declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`, `low_ctr_visible_page`) overlap on real pages in a messy way, shown below. Most flagged pages trigger only one flag, but thousands trigger two or more, and a handful trigger four. A fixed rule has to pick static weights for these flags up front and hope they generalize. A model can instead learn, from the data itself, how much each flag combination actually matters, and let that weighting shift with content type and position tier instead of staying fixed. That's a pattern that's real but too tangled for me to hand-write correctly, which is exactly the bar the framing skill sets for when ML earns its place over a plain rule.


In [5]:
flags = pd.DataFrame({
    "stale_visible_page": (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500),
    "declining_with_demand": (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100),
    "thin_visible_page": (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250),
    "page_one_decay_risk": (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180),
    "low_ctr_visible_page": (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5),
})

flag_count = flags.sum(axis=1)
print("how many of the 5 baseline flags each page triggers:")
print(flag_count.value_counts().sort_index())
print(f"\npages triggering 2+ overlapping flags: {(flag_count >= 2).sum()}")
print(f"pages triggering 0 flags (but may still be a candidate on other grounds): {(flag_count == 0).sum()}")


how many of the 5 baseline flags each page triggers:
0    10350
1    11212
2     6447
3     1984
4        7
Name: count, dtype: int64

pages triggering 2+ overlapping flags: 8438
pages triggering 0 flags (but may still be a candidate on other grounds): 10350


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.